# Network Formation Game (NFG) — Exp 1 & Exp 2 (Layered Graphs)

Integer-splittable, edge-capacitated network formation game with Shapley cost sharing.

**Cost function**: $\pi_i(X) = \sum_e w_e \cdot X_{i,e} / \ell_e$ (Shapley/proportional sharing)

## Layered Graph Design

- **L** layers × **W** nodes per layer, random inter-layer edges with p=0.8
- **Asymmetric only**: sources from layer 0, sinks from layer L-1

| | **Asymmetric** (distinct s-t from layer 0 / layer L-1) |
|---|---|
| **Tight capacity** | Set E |
| **Loose capacity** | Set F |

### Experiment 1: Best PNE and POS (BRD + GZR @ α=1)
- BRD warm-starts GZR (if PNE found); GZR optimises to MIP gap=0 (`stop_at_first=False`) to find the **best** (minimum social cost) PNE
- POS = best_pne_cost / so_cost
- Weaker VEST cut enabled (default): fundamental-cycle-basis cuts (Lemma 2 / Eq. 22)

### Experiment 2: Tightest Alpha Search (TAS)
- Only for instances where Exp1 GZR was infeasible or timed out (and BRD found no PNE)
- Bisection with verified/unverified bounds

**Note**: Experiment 1B (without weaker VEST cut) is in a separate notebook (`02b_run_nfg_layered_exp1b.ipynb`). Run in a fresh kernel after shutting down this notebook to avoid execution order bias.

In [ ]:
import numpy as np
import pandas as pd
import time
from pathlib import Path

from gipg.nfg.instance import NFGInstance
from gipg.nfg.objectives import player_cost, all_player_costs, social_cost, edge_loads
from gipg.nfg.best_response import solve_best_response
from gipg.nfg.gzr import solve_gzr
from gipg.nfg.heuristics import alpha_of_profile, brd_random_restart
from gipg.nfg.social_optimum import solve_social_optimum, compute_pos

RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('NFG modules loaded successfully.')

## 1. Load Shared Layered Instances

Load layered network data generated by `01_generate_layered_networks.ipynb`
(Sets E and F) and wrap as NFGInstance.

In [ ]:
import json, glob

SHARED_DIR = Path('../data/nfg')
assert SHARED_DIR.exists(), f'Shared instances not found at {SHARED_DIR}. Run 01_generate_layered_networks.ipynb first.'

# Load only layered instances (Set E, F)
json_files = sorted(glob.glob(str(SHARED_DIR / 'E_*.json'))) + \
             sorted(glob.glob(str(SHARED_DIR / 'F_*.json')))
print(f'Found {len(json_files)} layered network files')

instances = {}
n_loaded = 0

for fp in json_files:
    with open(fp) as f:
        d = json.load(f)

    tag = d['tag']
    edges_tuple = tuple(tuple(e) for e in d['edges'])

    inst = NFGInstance(
        n_nodes=d['n_nodes'],
        edges=edges_tuple,
        n_players=d['n_players'],
        sources=np.array(d['sources'], dtype=int),
        sinks=np.array(d['sinks'], dtype=int),
        demands=np.array(d['demands'], dtype=int),
        capacities=np.array(d['capacities'], dtype=int),
        edge_costs=np.array(d['edge_costs'], dtype=float),
        seed=d['seed'],
        meta=d['meta'],
    )
    instances[tag] = inst
    n_loaded += 1

print(f'Loaded {n_loaded} NFG instances (layered graphs)')

# Summary by set
SET_LABELS = {'E': 'tight+asym (layered)', 'F': 'loose+asym (layered)'}
for label, desc in SET_LABELS.items():
    count = sum(1 for t in instances if t.startswith(label + '_'))
    print(f'  Set {label} ({desc}): {count}')

# Capacity tightness summary
print('\n--- Capacity tightness summary ---')
for label in ['E', 'F']:
    subset = [inst for t, inst in instances.items() if t.startswith(label + '_')]
    if not subset:
        continue
    cap_ratios = [int(inst.capacities.min()) / int(inst.demands.sum())
                  for inst in subset if inst.demands.sum() > 0]
    if cap_ratios:
        print(f'  Set {label}: min(cap)/D  mean={np.mean(cap_ratios):.2f}  '
              f'range=[{np.min(cap_ratios):.2f}, {np.max(cap_ratios):.2f}]')

## 2. Experiment 1: Best PNE and POS (BRD + GZR @ alpha=1)

For each instance:
1. **BRD** (random-restart best-response dynamics) — fast heuristic to find a PNE candidate
2. **GZR @ alpha=1** with `stop_at_first=False` — optimises to MIP gap=0, finding the **best** PNE (minimum social cost). Warm-started with BRD profile if BRD found a PNE.
3. **Social Optimum** — minimum social cost (no equilibrium constraints)
4. **POS** = best_pne_cost / so_cost

In [ ]:
TOTAL_TIME_LIMIT = 600.0  # Total budget for BRD + GZR combined
SO_TIME_LIMIT = 1200.0    # 20 minutes for social optimum (reference computation)

results_exp1 = []

for idx, (tag, inst) in enumerate(instances.items()):
    cap_level = inst.meta.get('capacity_level', 'unknown')
    sym = inst.meta.get('symmetry', 'unknown')

    row = {
        'tag': tag,
        'graph_type': inst.meta.get('graph_type', 'layered'),
        'capacity_level': cap_level,
        'symmetry': sym,
        'n_players': inst.n_players,
        'n_nodes': inst.n_nodes,
        'n_edges': inst.n_edges,
        'n_layers': inst.meta.get('n_layers', None),
        'n_width': inst.meta.get('n_width', None),
        'seed': inst.meta.get('seed', None),
    }

    # --- Phase 1: BRD ---
    x_brd, brd_pne, brd_time = brd_random_restart(
        inst, max_init=3, max_round=15, seed=0,
    )
    row['brd_found_pne'] = brd_pne
    row['brd_time'] = brd_time
    row['brd_alpha'] = alpha_of_profile(inst, x_brd) if brd_pne else float('inf')
    if brd_pne:
        row['initial_pne_cost'] = social_cost(inst, x_brd)

    # --- Phase 2: GZR @ alpha=1, warm-started with BRD profile ---
    # GZR time limit = remaining budget after BRD (min 60s safety floor)
    gzr_time_limit = max(TOTAL_TIME_LIMIT - brd_time, 60.0)

    warm = x_brd if brd_pne else None
    gzr_res = solve_gzr(
        inst, alpha=1.0, time_limit=gzr_time_limit,
        warm_start=warm,
        stop_at_first=False, verbose=False,
    )
    row['gzr_status'] = gzr_res.status
    row['gzr_mip_gap'] = gzr_res.mip_gap
    row['gzr_obj_bound'] = gzr_res.obj_bound
    row['gzr_cuts'] = gzr_res.cuts_added
    row['gzr_br_calls'] = gzr_res.br_calls
    row['gzr_time'] = brd_time + gzr_res.runtime   # BRD + GZR combined
    row['gzr_obj_val'] = gzr_res.obj_val
    row['gzr_first_pne_time'] = (brd_time + gzr_res.first_pne_time
                                        if gzr_res.first_pne_time is not None else None)

    if gzr_res.profile is not None:
        row['best_pne_cost'] = social_cost(inst, gzr_res.profile)
    elif brd_pne:
        row['best_pne_cost'] = row['initial_pne_cost']
    else:
        row['best_pne_cost'] = None

    # --- Phase 3: Social Optimum ---
    so_res = solve_social_optimum(inst, time_limit=SO_TIME_LIMIT, verbose=False)
    row['so_status'] = so_res.status
    row['so_cost'] = so_res.opt_cost
    row['so_time'] = so_res.runtime

    # --- POS ---
    row['pos'] = None
    if row['best_pne_cost'] is not None and so_res.opt_cost is not None and so_res.opt_cost > 0:
        row['pos'] = compute_pos(row['best_pne_cost'], so_res.opt_cost)

    results_exp1.append(row)

    # Progress
    pne_str = 'BRD-PNE' if brd_pne else 'no-PNE'
    pos_str = f"POS={row['pos']:.3f}" if row['pos'] is not None else 'POS=N/A'
    fpne_str = f"1stPNE={row['gzr_first_pne_time']:.1f}s" if row.get('gzr_first_pne_time') is not None else '1stPNE=N/A'
    print(f'[{idx+1}/{len(instances)}] {tag}: {pne_str} | GZR {gzr_res.status} '
          f'({row["gzr_time"]:.1f}s, {gzr_res.cuts_added}cuts) | {pos_str} | {fpne_str}')

df_exp1 = pd.DataFrame(results_exp1)
df_exp1.to_csv(RESULTS_DIR / 'nfg_layered_exp1_brd_gzr_pos.csv', index=False)
print(f'\nSaved {len(df_exp1)} rows to nfg_layered_exp1_brd_gzr_pos.csv')

In [ ]:
# Experiment 1 Summary
print('=== Experiment 1 Summary (Layered NFG) ===\n')

n_total = len(df_exp1)
n_brd_pne = df_exp1['brd_found_pne'].sum()
n_gzr_opt = (df_exp1['gzr_status'] == 'OPTIMAL').sum()
n_gzr_inf = (df_exp1['gzr_status'] == 'INFEASIBLE').sum()
n_gzr_tl = (df_exp1['gzr_status'] == 'TIME_LIMIT').sum()
n_pos = df_exp1['pos'].notna().sum()

print(f'Total instances: {n_total}')
print(f'BRD found PNE: {n_brd_pne} ({n_brd_pne/n_total:.1%})')
print(f'GZR OPTIMAL: {n_gzr_opt} ({n_gzr_opt/n_total:.1%})')
print(f'GZR INFEASIBLE: {n_gzr_inf} ({n_gzr_inf/n_total:.1%})')
print(f'GZR TIME_LIMIT: {n_gzr_tl} ({n_gzr_tl/n_total:.1%})')
print(f'POS computed: {n_pos} ({n_pos/n_total:.1%})')

# By capacity_level
print('\n--- By capacity_level ---')
summary = df_exp1.groupby(['capacity_level']).agg(
    n=('tag', 'count'),
    brd_pne_rate=('brd_found_pne', 'mean'),
    gzr_opt_rate=('gzr_status', lambda x: (x == 'OPTIMAL').mean()),
    gzr_inf_rate=('gzr_status', lambda x: (x == 'INFEASIBLE').mean()),
    avg_gzr_time=('gzr_time', 'mean'),
    avg_cuts=('gzr_cuts', 'mean'),
    avg_pos=('pos', 'mean'),
    max_pos=('pos', 'max'),
).round(4)
print(summary)

# By (n_layers, n_width)
print('\n--- By (n_layers, n_width) ---')
summary_lw = df_exp1.groupby(['n_layers', 'n_width']).agg(
    n=('tag', 'count'),
    n_nodes_mean=('n_nodes', 'mean'),
    n_edges_mean=('n_edges', 'mean'),
    gzr_opt_rate=('gzr_status', lambda x: (x == 'OPTIMAL').mean()),
    avg_gzr_time=('gzr_time', 'mean'),
    avg_cuts=('gzr_cuts', 'mean'),
    avg_pos=('pos', 'mean'),
    max_pos=('pos', 'max'),
).round(4)
print(summary_lw)

# POS distribution
pos_valid = df_exp1[df_exp1['pos'].notna()]
if len(pos_valid) > 0:
    print(f'\nPOS statistics (n={len(pos_valid)}):')
    print(f'  Mean: {pos_valid["pos"].mean():.4f}')
    print(f'  Max:  {pos_valid["pos"].max():.4f}')
    print(f'  Min:  {pos_valid["pos"].min():.4f}')

## 3. Experiment 2: Tightest Alpha Search (TAS) for Non-PNE Instances

Only instances where Exp1 GZR reported **INFEASIBLE** or **TIME_LIMIT** (and BRD found no PNE).

Bisection with verified/unverified bounds (ISBP pattern).

In [ ]:
TAS_GZR_LIMIT = 1200.0   # 20 minutes per GZR call in TAS
TAS_EPS = 1e-2
TAS_MAX_ITERS = 30

failed_tags = [
    r['tag'] for r in results_exp1
    if r['gzr_status'] in ('INFEASIBLE', 'TIME_LIMIT') and not r.get('brd_found_pne', False)
]
print(f'Exp2 candidates: {len(failed_tags)} instances')

results_exp2 = []

for idx, tag in enumerate(failed_tags):
    inst = instances[tag]
    start_time = time.time()
    cap_level = inst.meta.get('capacity_level', 'unknown')
    sym = inst.meta.get('symmetry', 'unknown')

    row = {
        'tag': tag,
        'graph_type': inst.meta.get('graph_type', 'layered'),
        'capacity_level': cap_level,
        'symmetry': sym,
        'n_players': inst.n_players,
        'n_nodes': inst.n_nodes,
        'n_edges': inst.n_edges,
        'seed': inst.meta.get('seed', None),
    }

    # --- Phase 0: BRD upper bound on alpha ---
    x_brd, brd_pne, _ = brd_random_restart(
        inst, max_init=5, max_round=20, seed=0,
    )
    alpha_init = alpha_of_profile(inst, x_brd)
    ub_verified = alpha_init
    best_profile = x_brd

    # --- Phase 1: Reuse Exp1 result for alpha=1 ---
    exp1_row = next(r for r in results_exp1 if r['tag'] == tag)
    if exp1_row['gzr_status'] == 'INFEASIBLE':
        lb_verified = 1.0
        lb_unverified = 1.0
    else:  # TIME_LIMIT
        lb_verified = None
        lb_unverified = 1.0

    # --- Phase 2: Bisection ---
    n_iters = 0
    history = []

    while (ub_verified - lb_unverified > TAS_EPS) and (n_iters < TAS_MAX_ITERS):
        n_iters += 1
        alpha_mid = 0.5 * (lb_unverified + ub_verified)

        gzr_mid = solve_gzr(
            inst, alpha=alpha_mid, time_limit=TAS_GZR_LIMIT,
            verbose=False,
        )

        if gzr_mid.status == 'FEASIBLE' and gzr_mid.profile is not None:
            ub_verified = alpha_mid
            best_profile = gzr_mid.profile
            bound_type = 'verified_ub'
        elif gzr_mid.status == 'INFEASIBLE':
            lb_unverified = alpha_mid
            lb_verified = alpha_mid if lb_verified is None else max(lb_verified, alpha_mid)
            bound_type = 'verified_lb'
        else:
            lb_unverified = alpha_mid
            bound_type = 'unverified_lb'

        history.append({
            'iter': n_iters, 'alpha_mid': alpha_mid,
            'status': gzr_mid.status, 'bound_type': bound_type,
            'runtime': gzr_mid.runtime, 'cuts': gzr_mid.cuts_added,
        })
        print(f'  iter {n_iters}: alpha={alpha_mid:.4f} -> {gzr_mid.status} ({bound_type}) '
              f'[{gzr_mid.runtime:.1f}s]')

    row['alpha_init'] = alpha_init
    row['alpha_star'] = ub_verified
    row['alpha_lb_verified'] = lb_verified
    row['alpha_lb_unverified'] = lb_unverified
    row['alpha_ub_verified'] = ub_verified
    row['n_bisect_iters'] = n_iters
    row['tas_time_total'] = time.time() - start_time
    results_exp2.append(row)

    print(f'[{idx+1}/{len(failed_tags)}] {tag}: alpha*={ub_verified:.4f} '
          f'(lb_v={lb_verified}, lb_u={lb_unverified:.4f}) '
          f'{n_iters} iters, {row["tas_time_total"]:.1f}s')

df_exp2 = pd.DataFrame(results_exp2)
if len(df_exp2) > 0:
    df_exp2.to_csv(RESULTS_DIR / 'nfg_layered_exp2_tas.csv', index=False)
    print(f'\nSaved {len(df_exp2)} rows to nfg_layered_exp2_tas.csv')
else:
    print('\nNo instances needed TAS (all solved in Exp1).')

In [ ]:
# Experiment 2 Summary
print('=== Experiment 2: TAS Summary (Layered NFG) ===\n')
if len(df_exp2) > 0:
    print(f'Instances in TAS: {len(df_exp2)}')
    print(f'Mean alpha*: {df_exp2["alpha_star"].mean():.4f}')
    print(f'Max  alpha*: {df_exp2["alpha_star"].max():.4f}')
    n_verified_lb = df_exp2['alpha_lb_verified'].notna().sum()
    print(f'Instances with verified lower bound: {n_verified_lb} ({n_verified_lb/len(df_exp2):.1%})')
    print(f'Mean bisection iters: {df_exp2["n_bisect_iters"].mean():.1f}')
    print(f'Mean total time: {df_exp2["total_time"].mean():.1f}s')

    print('\n--- By capacity_level ---')
    summary2 = df_exp2.groupby(['capacity_level']).agg(
        n=('tag', 'count'),
        avg_alpha_star=('alpha_star', 'mean'),
        max_alpha_star=('alpha_star', 'max'),
        pct_verified_lb=('alpha_lb_verified', lambda x: x.notna().mean()),
        avg_iters=('n_bisect_iters', 'mean'),
        avg_time=('total_time', 'mean'),
    ).round(4)
    print(summary2)
else:
    print('No instances required TAS — all solved in Exp1.')

## 4. Combined Results

In [ ]:
# Merge Exp1 and Exp2
if len(df_exp2) > 0:
    exp2_cols = ['tag', 'alpha_init', 'alpha_star', 'alpha_lb_verified',
                 'alpha_lb_unverified', 'alpha_ub_verified', 'n_bisect_iters']
    df_all = pd.merge(df_exp1, df_exp2[exp2_cols], on='tag', how='left')
else:
    df_all = df_exp1.copy()
    for col in ['alpha_init', 'alpha_star', 'alpha_lb_verified',
                'alpha_lb_unverified', 'alpha_ub_verified', 'n_bisect_iters']:
        df_all[col] = np.nan

# For instances solved in Exp1, alpha_star = 1.0
mask_solved = (df_all['gzr_status'] == 'OPTIMAL') | (df_all['brd_found_pne'] == True)
df_all.loc[mask_solved & df_all['alpha_star'].isna(), 'alpha_star'] = 1.0

df_all.to_csv(RESULTS_DIR / 'nfg_layered_results_all.csv', index=False)
print(f'Combined results: {len(df_all)} rows -> nfg_layered_results_all.csv')
print(f'Columns: {list(df_all.columns)}')

# Final summary
print('\n=== Final Summary (Layered NFG) ===')
for cap in ['tight', 'loose']:
    sub = df_all[df_all['capacity_level'] == cap]
    if len(sub) == 0:
        continue
    n_pne = ((sub['gzr_status'] == 'OPTIMAL') | (sub['brd_found_pne'] == True)).sum()
    pos_ok = sub['pos'].notna()
    pos_str = f'POS mean={sub.loc[pos_ok, "pos"].mean():.3f}' if pos_ok.any() else 'no POS'
    tas_sub = sub[sub['n_bisect_iters'].notna() & (sub['n_bisect_iters'] > 0)]
    tas_str = f'TAS: {len(tas_sub)} inst' if len(tas_sub) > 0 else 'no TAS'
    print(f'  {cap}+asym (layered): {len(sub)} inst, PNE={n_pne}, {pos_str}, {tas_str}')